In [ ]:
# ¿Cómo podemos anticipar si un tumor de mama es maligno o benigno a partir de medidas numéricas de sus células?

# El archivo contiene medidas de imágenes de biopsias con su diagnóstico observado.

import pandas as pd

dataframe = pd.read_csv("../data/wisc_bc_data.csv")

In [ ]:
# ¿Cuántos casos y variables están disponibles para entrenar y evaluar el clasificador?

dataframe.shape

In [ ]:
# Se inspeccionan las variables que describen cada caso y su diagnóstico observado.

dataframe.head()

In [ ]:
# ¿Hay valores faltantes?

dataframe.isna().sum().sum()

In [ ]:
# ¿Las clases de diagnóstico están balanceadas? (B = benigno, M = maligno)

dataframe.diagnosis.value_counts()

In [ ]:
# ¿Las medidas son distintas entre tumores benignos y malignos?

dataframe.groupby("diagnosis")[["radius_mean", "texture_mean", "concave points_mean"]].mean()

In [ ]:
# El identificador no describe el tumor, por eso no se usa como variable explicativa.

features = dataframe.drop(columns=["id", "diagnosis"])
target = dataframe["diagnosis"]

features.shape

In [ ]:
# Se reservan casos de cada diagnóstico para evaluar predicciones sobre datos no vistos.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    random_state=0,
    stratify=target,
)

len(X_train), len(X_test)

In [ ]:
# Las variables tienen escalas muy distintas (por ejemplo, area_mean frente a smoothness_mean).

X_train[["area_mean", "smoothness_mean"]].describe().transpose()[["mean", "std"]]

In [ ]:
# Modelo base: regresión logística con todas las variables estandarizadas.
#
# El estimador es un Pipeline: guarda el escalamiento y el clasificador juntos,
# así el archivo almacenado basta para predecir casos nuevos.

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

base_estimator = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000)),
    ],
)

base_estimator.fit(X_train, y_train)

In [ ]:
# Se compara el desempeño de entrenamiento y prueba para detectar una posible brecha de generalización.

from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score

# Precisión de entrenamiento:

accuracy_score(
    y_true=y_train,
    y_pred=base_estimator.predict(X_train),
)

In [ ]:
# La precisión por sí sola puede ocultar fallas en la clase menos frecuente.
# El recall de la clase M mide qué proporción de tumores malignos detecta el modelo.

test_results = {}

base_predictions = base_estimator.predict(X_test)

test_results["base_estimator"] = {
    "train_accuracy": accuracy_score(y_true=y_train, y_pred=base_estimator.predict(X_train)),
    "test_accuracy": accuracy_score(y_true=y_test, y_pred=base_predictions),
    "test_balanced_accuracy": balanced_accuracy_score(y_true=y_test, y_pred=base_predictions),
    "test_recall_malignant": recall_score(y_true=y_test, y_pred=base_predictions, pos_label="M"),
}

test_results

In [ ]:
# La matriz muestra qué clases confunde el modelo. En este problema, el error más costoso
# es clasificar como benigno (B) un tumor maligno (M).

from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay.from_predictions(
    y_test,
    base_predictions,
    cmap="Greens",
)
disp.figure_.suptitle("Matriz de confusión - modelo base")
print(f"Matriz de confusión:\n{disp.confusion_matrix}")

In [ ]:
# Modelo flexible: red neuronal con dos capas ocultas, capaz de aprender fronteras no lineales.

from sklearn.neural_network import MLPClassifier

flexible_estimator = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            MLPClassifier(
                max_iter=10000,
                hidden_layer_sizes=(64, 64),
                activation="relu",
                solver="adam",
                learning_rate_init=0.001,
                validation_fraction=0.2,
                early_stopping=True,
                random_state=0,
            ),
        ),
    ],
)

flexible_estimator.fit(X_train, y_train)

In [ ]:
# Se evalúa el modelo flexible con las mismas métricas del modelo base.

flexible_predictions = flexible_estimator.predict(X_test)

test_results["flexible_estimator"] = {
    "train_accuracy": accuracy_score(y_true=y_train, y_pred=flexible_estimator.predict(X_train)),
    "test_accuracy": accuracy_score(y_true=y_test, y_pred=flexible_predictions),
    "test_balanced_accuracy": balanced_accuracy_score(y_true=y_test, y_pred=flexible_predictions),
    "test_recall_malignant": recall_score(y_true=y_test, y_pred=flexible_predictions, pos_label="M"),
}

disp = ConfusionMatrixDisplay.from_predictions(
    y_test,
    flexible_predictions,
    cmap="Greens",
)
disp.figure_.suptitle("Matriz de confusión - modelo flexible")
print(f"Matriz de confusión:\n{disp.confusion_matrix}")

In [ ]:
# Desempeño de los modelos. Un modelo más flexible no siempre generaliza mejor.

model_comparison = pd.DataFrame(test_results).T
model_comparison

In [ ]:
# Se guardan los estimadores, las métricas y la comparación como evidencia verificable.

import json
import pickle

with open("../submission/base_estimator.pkl", "wb") as file:
    pickle.dump(base_estimator, file)

with open("../submission/flexible_estimator.pkl", "wb") as file:
    pickle.dump(flexible_estimator, file)

metrics = {
    "base_estimator": test_results["base_estimator"],
    "flexible_estimator": test_results["flexible_estimator"],
    "test_size": len(y_test),
}

with open("../submission/metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

model_comparison.to_csv("../submission/model_comparison.csv", index_label="model")

In [ ]:
# Se comprueba que el estimador almacenado predice y entrega la probabilidad de malignidad de cada caso.
# Las clases quedan en orden alfabético (B, M), por eso la columna 1 de predict_proba es la de M.

with open("../submission/base_estimator.pkl", "rb") as file:
    new_estimator = pickle.load(file)

new_predictions = new_estimator.predict(X_test)
new_predicted_proba = new_estimator.predict_proba(X_test)

results = pd.DataFrame(
    {
        "observed_diagnosis": y_test.reset_index(drop=True),
        "predicted_diagnosis": new_predictions,
        "malignant_probability": new_predicted_proba[:, 1],
    }
)
results["malignant_probability"] = results["malignant_probability"].round(3)
results.head(10)